<a href="https://colab.research.google.com/github/Arjunanand717/Deep-learing-/blob/main/EXP_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Resources

**Name:** ARJUN S
**Registration No:** 24BAD401

## Loading Yelp Polarity Reviews Dataset

In [ ]:
# Install the datasets library if you haven't already
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("fancyzhx/yelp_polarity")

print(ds)

README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  256MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 17.7MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})


In [ ]:
# Display a sample of the dataset
print("\nSample from the training split:")
print(ds['train'][0])

print("\nSample from the test split:")
print(ds['test'][0])


Sample from the training split:
{'text': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars.", 'label': 0}

Sample from the test split:
{'text': "Contrary to other reviews, I have zero complaints about the service or the prices. I have been getting tire service here for the past 5 years now, and compared to my experience with places like Pep Boys, these guys are experienced and know what they're doing. 

### 1. Extract text and labels from the Yelp dataset

In [ ]:
# Extract texts and labels from the 'ds' object
yelp_train_texts = [item['text'] for item in ds['train']]
yelp_train_labels = [item['label'] for item in ds['train']]
yelp_test_texts = [item['text'] for item in ds['test']]
yelp_test_labels = [item['label'] for item in ds['test']]

print(f"Number of training reviews: {len(yelp_train_texts)}")
print(f"Number of testing reviews: {len(yelp_test_texts)}")
print(f"Sample training text: {yelp_train_texts[0][:200]}...")
print(f"Sample training label: {yelp_train_labels[0]}")

Number of training reviews: 560000
Number of testing reviews: 38000
Sample training text: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply ...
Sample training label: 0


### 2. Tokenize and convert text to numerical sequences

We will use Keras' `Tokenizer` to convert the text reviews into sequences of integers. This involves fitting the tokenizer on the training data to build a vocabulary, then converting both training and test texts to sequences.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Define the maximum number of words to keep in the vocabulary
# This is similar to num_words in imdb.load_data
YELP_VOCAB_SIZE = 10000

tokenizer = Tokenizer(num_words=YELP_VOCAB_SIZE, oov_token='<unk>')

# Fit tokenizer on training texts
tokenizer.fit_on_texts(yelp_train_texts)

# Convert texts to sequences of integers
yelp_train_sequences = tokenizer.texts_to_sequences(yelp_train_texts)
yelp_test_sequences = tokenizer.texts_to_sequences(yelp_test_texts)

print(f"Sample original text: {yelp_train_texts[0][:100]}...")
print(f"Sample tokenized sequence: {yelp_train_sequences[0][:10]}...")

Sample original text: Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've ha...
Sample tokenized sequence: [708, 2, 4340, 8, 174, 940, 1, 1666, 12, 6]...


### 3. Apply sequence padding

As before, we need to pad the sequences to ensure all input sequences have the same length for the LSTM model. We'll reuse the `MAXLEN` we defined earlier, or determine a new one suitable for Yelp reviews.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Let's check average and max review length for Yelp data
yelp_review_lengths = [len(seq) for seq in yelp_train_sequences]
print(f"Yelp Average review length: {np.mean(yelp_review_lengths):.2f}")
print(f"Yelp Max review length: {np.max(yelp_review_lengths)}")

# We can choose a MAXLEN based on this distribution or reuse the IMDB MAXLEN.
# Let's use MAXLEN = 250 as it's a reasonable compromise.
# If you want to use a specific MAXLEN for Yelp, you can define YELP_MAXLEN here.

yelp_train_padded = pad_sequences(yelp_train_sequences, maxlen=MAXLEN, padding='post')
yelp_test_padded = pad_sequences(yelp_test_sequences, maxlen=MAXLEN, padding='post')

print(f"\nShape of yelp_train_padded: {yelp_train_padded.shape}")
print(f"Shape of yelp_test_padded: {yelp_test_padded.shape}")
print(f"Sample of padded Yelp review (first 10 elements): {yelp_train_padded[0][:10]}")

Yelp Average review length: 136.22
Yelp Max review length: 1046

Shape of yelp_train_padded: (560000, 250)
Shape of yelp_test_padded: (38000, 250)
Sample of padded Yelp review (first 10 elements): [ 708    2 4340    8  174  940    1 1666   12    6]


### 4. Convert labels to NumPy arrays and split into training/validation

The labels are already numerical (0 or 1), so we just need to convert them to NumPy arrays and then split the training data for validation.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

yelp_train_labels = np.array(yelp_train_labels)
yelp_test_labels = np.array(yelp_test_labels)

# Create a validation set from the Yelp training data
X_yelp_train, X_yelp_val, Y_yelp_train, Y_yelp_val = train_test_split(
    yelp_train_padded, yelp_train_labels, test_size=0.2, random_state=42
)

print(f"\nShape of X_yelp_train: {X_yelp_train.shape}")
print(f"Shape of X_yelp_val: {X_yelp_val.shape}")
print(f"Shape of yelp_test_padded: {yelp_test_padded.shape}")
print(f"Shape of Y_yelp_train: {Y_yelp_train.shape}")
print(f"Shape of Y_yelp_val: {Y_yelp_val.shape}")
print(f"Shape of yelp_test_labels: {yelp_test_labels.shape}")


Shape of X_yelp_train: (448000, 250)
Shape of X_yelp_val: (112000, 250)
Shape of yelp_test_padded: (38000, 250)
Shape of Y_yelp_train: (448000,)
Shape of Y_yelp_val: (112000,)
Shape of yelp_test_labels: (38000,)


## Re-training with Yelp Dataset (Optional: Re-build and Re-train Model)

If you want to train a new model specifically for the Yelp dataset, you would rebuild the model using `YELP_VOCAB_SIZE` and then train it with `X_yelp_train`, `Y_yelp_train`, etc. Otherwise, the existing model was trained on IMDB data.

### Build and compile the LSTM model for Yelp data

We will reuse the same LSTM architecture but ensure it's built for the `YELP_VOCAB_SIZE`.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Model Parameters (using Yelp-specific vocabulary size)
# YELP_VOCAB_SIZE = 10000  # defined earlier
# EMBEDDING_DIM = 128     # can be reused
# MAXLEN = 250            # can be reused

# Build the new LSTM model for Yelp
yelp_model = Sequential([
    Embedding(input_dim=YELP_VOCAB_SIZE + 1, output_dim=EMBEDDING_DIM, input_length=MAXLEN),
    LSTM(128), # LSTM layer with 128 units
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
yelp_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display the model architecture and summary
yelp_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train the LSTM model on Yelp data

We will train the new model using the prepared Yelp training dataset (`X_yelp_train`, `Y_yelp_train`) and validate it using `X_yelp_val`, `Y_yelp_val`.

In [ ]:
EPOCHS = 1 # Reduced to 1 epoch for minimal train and test
BATCH_SIZE = 64 # You can adjust this value

yelp_history = yelp_model.fit(
    X_yelp_train,
    Y_yelp_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_yelp_val, Y_yelp_val)
)

print("\nYelp Model training complete.")

1550/7000 ━━━━━━━━━━━━━━━━━━━━ 51:55 572ms/step - accuracy: 0.9557 - loss: 0.1178

KeyboardInterrupt: 

### Record and visualize Yelp model training history

In [26]:
import matplotlib.pyplot as plt

# Get training and validation metrics for Yelp model
yelp_acc = yelp_history.history['accuracy']
yelp_val_acc = yelp_history.history['val_accuracy']
yelp_loss = yelp_history.history['loss']
yelp_val_loss = yelp_history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, yelp_acc, label='Training Accuracy')
plt.plot(epochs_range, yelp_val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Yelp Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, yelp_loss, label='Training Loss')
plt.plot(epochs_range, yelp_val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Yelp Training and Validation Loss')

plt.show()

NameError: name 'yelp_history' is not defined

### Evaluate the trained Yelp model using the test dataset

In [ ]:
yelp_loss, yelp_accuracy = yelp_model.evaluate(yelp_test_padded, yelp_test_labels, verbose=1)

print(f"\nYelp Test Loss: {yelp_loss:.4f}")
print(f"Yelp Test Accuracy: {yelp_accuracy:.4f}")

1188/1188 ━━━━━━━━━━━━━━━━━━━━ 175s 147ms/step - accuracy: 0.9490 - loss: 0.1350

Yelp Test Loss: 0.1350
Yelp Test Accuracy: 0.9490


### Predict sentiment for Yelp reviews and calculate evaluation metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the Yelp test set
yelp_pred_probs = yelp_model.predict(yelp_test_padded)
yelp_pred = (yelp_pred_probs > 0.5).astype(int)

# Print classification report
print("\nYelp Classification Report:")
print(classification_report(yelp_test_labels, yelp_pred))

# Print confusion matrix
print("\nYelp Confusion Matrix:")
print(confusion_matrix(yelp_test_labels, yelp_pred))

## Part A: Dataset Preparation and Text Preprocessing

### 1. Load the IMDB Movie Reviews dataset

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Load the IMDB dataset
# num_words parameter keeps only the top N most frequently occurring words
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

print(f"Training samples: {len(x_train)}")
print(f"Testing samples: {len(x_test)}")
print(f"Labels: {np.unique(y_train)}") # 0 for negative, 1 for positive

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training samples: 25000
Testing samples: 25000
Labels: [0 1]


### 2. Explore the dataset and examine sample reviews

The `imdb.load_data` function returns sequences of integers, where each integer represents a word. To view the actual text reviews, we need to map these integers back to words using the IMDb word index dictionary.

In [ ]:
# Get the word index dictionary
word_index = imdb.get_word_index()

# Create a reverse word index (mapping from integer to word)
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Decode a sample review (e.g., the first training review)
# The indices are offset by 3 because 0, 1, and 2 are reserved indices for "padding", "start of sequence", and "unknown"
def decode_review(text_sequence):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text_sequence])

print("\nSample Positive Review:")
print(decode_review(x_train[0]))
print(f"Label: {y_train[0]} (Positive)")

print("\nSample Negative Review:")
print(decode_review(x_train[1]))
print(f"Label: {y_train[1]} (Negative)")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Sample Positive Review:
? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these childr

### 3. Check sentiment class distribution

We can see the distribution of positive (1) and negative (0) reviews in both the training and testing datasets.

In [ ]:
import collections

train_label_counts = collections.Counter(y_train)
test_label_counts = collections.Counter(y_test)

print(f"Training label distribution: {train_label_counts}")
print(f"Testing label distribution: {test_label_counts}")

# This shows a balanced dataset with 12500 positive and 12500 negative reviews in both train and test sets.

Training label distribution: Counter({np.int64(1): 12500, np.int64(0): 12500})
Testing label distribution: Counter({np.int64(0): 12500, np.int64(1): 12500})


### 4. Text preprocessing using NLP libraries

The `imdb.load_data` function already handles much of the preprocessing, including tokenization and converting text to numerical sequences, and implicitly lowercasing and removing punctuation. However, for a more explicit demonstration of `Tokenizer` and `pad_sequences` as requested, we can simulate these steps by using the raw text data (if available) or by focusing on `pad_sequences` for the numerical data we already have.

Since `imdb.load_data` already provides numerical sequences of words and implicitly handles lowercasing and punctuation removal (by only including words in its vocabulary), the primary remaining preprocessing step (beyond what `imdb.load_data` already does) for the numerical sequences is **sequence padding** to ensure all input sequences to the LSTM model have the same length.

For `remove stop words`, this is often handled by `num_words` in `imdb.load_data` which implicitly prioritizes more informative words by frequency. However, if a custom tokenizer were used, explicit stop word removal would be done.

Let's apply sequence padding.

#### Apply sequence padding

LSTM networks typically require input sequences to have a uniform length. `pad_sequences` adds zeros to the beginning or end of sequences to achieve this uniform length. We'll choose a `maxlen` (maximum sequence length) to ensure all reviews are the same length. Reviews longer than `maxlen` will be truncated, and shorter ones will be padded.

In [ ]:
# Determine a suitable maximum sequence length (maxlen)
# A common approach is to look at the distribution of review lengths
review_lengths = [len(x) for x in x_train]
print(f"Average review length: {np.mean(review_lengths):.2f}")
print(f"Max review length: {np.max(review_lengths)}")
print(f"Min review length: {np.min(review_lengths)}")

# Let's choose a maxlen, for instance, 250 words.
# This will truncate longer reviews and pad shorter ones.
MAXLEN = 250

# Pad sequences to ensure uniform length
x_train_padded = pad_sequences(x_train, maxlen=MAXLEN)
x_test_padded = pad_sequences(x_test, maxlen=MAXLEN)

print(f"\nShape of x_train after padding: {x_train_padded.shape}")
print(f"Shape of x_test after padding: {x_test_padded.shape}")
print(f"Sample of padded review (first 10 elements): {x_train_padded[0][:10]}")

Average review length: 238.71
Max review length: 2494
Min review length: 11

Shape of x_train after padding: (25000, 250)
Shape of x_test after padding: (25000, 250)
Sample of padded review (first 10 elements): [0 0 0 0 0 0 0 0 0 0]


#### Encode sentiment labels into numerical form

Our labels `y_train` and `y_test` are already in numerical form (0 for negative, 1 for positive), so no explicit encoding step is needed here.

In [ ]:
print(f"Labels are already numerical: y_train[:5] = {y_train[:5]}")
print(f"Labels are already numerical: y_test[:5] = {y_test[:5]}")

Labels are already numerical: y_train[:5] = [1 0 0 1 0]
Labels are already numerical: y_test[:5] = [0 1 1 0 1]


### 5. Split the prepared dataset into training and testing/validation datasets

The `imdb.load_data` function already provides a pre-split training and testing dataset. For a validation set, we can further split the training data.

In [ ]:
from sklearn.model_selection import train_test_split

# We already have x_train_padded, y_train, x_test_padded, y_test
# Let's create a validation set from the training data

X_train, X_val, Y_train, Y_val = train_test_split(x_train_padded, y_train, test_size=0.2, random_state=42)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of x_test_padded (for final evaluation): {x_test_padded.shape}")
print(f"Shape of Y_train: {Y_train.shape}")
print(f"Shape of Y_val: {Y_val.shape}")
print(f"Shape of y_test (for final evaluation): {y_test.shape}")


Shape of X_train: (20000, 250)
Shape of X_val: (5000, 250)
Shape of x_test_padded (for final evaluation): (25000, 250)
Shape of Y_train: (20000,)
Shape of Y_val: (5000,)
Shape of y_test (for final evaluation): (25000,)


## Part B: Embedding Generation

### 1. Create an Embedding layer using Keras/TensorFlow

An Embedding layer is used to create word embeddings. It takes integers (our word indices) and maps them to dense vectors of a fixed size. Each word will get its own dense vector representation.

We need to define:
*   `input_dim`: The size of the vocabulary (number of unique words + 1 for padding/unknown words).
*   `output_dim`: The dimension of the dense embedding (e.g., 100, 128, 256).
*   `input_length`: The length of input sequences (our `MAXLEN`).

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.models import Sequential

# Define vocabulary size (num_words used when loading IMDB data + 1 for padding)
VOCAB_SIZE = 10000 + 1 # 10000 words + 1 for the '0' used in padding or out-of-vocabulary words

# Define embedding dimension
EMBEDDING_DIM = 128 # This can be tuned, common values are 100, 128, 256

# Create a simple Sequential model just to demonstrate the Embedding layer
embedding_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAXLEN)
])

embedding_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### 2. Convert tokenized sequences into dense vector representations

To see the output of the Embedding layer, we can pass a sample of our padded training data through this model.

In [ ]:
# Take a batch of padded sequences from X_train
sample_input = X_train[:1]

# Get the embedding output
embedded_output = embedding_model.predict(sample_input)

print(f"Shape of sample_input: {sample_input.shape}")
print(f"Shape of embedded_output: {embedded_output.shape}")

# The output shape is (batch_size, input_length, embedding_dim)
# For our sample: (1, {MAXLEN}, {EMBEDDING_DIM})

print("\nSample of embedded representation for the first word in the first review (first 5 dimensions):")
print(embedded_output[0][0][:5]) # First review, first word (padded 0), first 5 dimensions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
Shape of sample_input: (1, 250)
Shape of embedded_output: (1, 250, 128)

Sample of embedded representation for the first word in the first review (first 5 dimensions):
[ 0.0289746   0.01894889 -0.01025714 -0.03952922 -0.00076992]


### 3. Analyze how words are represented in the embedding space

The Embedding layer initializes random vectors for each word. During the training of the full LSTM model, these vectors will be adjusted so that words with similar meanings or contexts in the reviews will have similar vector representations. This is a powerful feature as it captures semantic relationships between words.

Since this is a randomly initialized embedding layer *before* training, the current values are arbitrary. The true power will be seen after the LSTM model is trained.

## Part C: Implementing the LSTM Model

### 1. Build and compile the LSTM model

We will construct a sequential Keras model with the following layers:
*   **Input Layer**: Defines the shape of the input sequences (our `MAXLEN`).
*   **Embedding Layer**: Maps the integer-encoded words to dense vectors of fixed size (`EMBEDDING_DIM`). This layer will learn meaningful word representations during training.
*   **LSTM Layer**: A Long Short-Term Memory layer, which is a type of recurrent neural network capable of learning long-term dependencies in sequence data. We'll use a specified number of LSTM units (e.g., 128).
*   **Dense Output Layer**: A fully connected layer with a single neuron and a 'sigmoid' activation function for binary classification (positive/negative sentiment).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Model Parameters (defined in Part B, but good to have here for context)
# VOCAB_SIZE = 10000 + 1 # defined earlier
# EMBEDDING_DIM = 128   # defined earlier
# MAXLEN = 250          # defined earlier

# Build the LSTM model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAXLEN),
    LSTM(128), # LSTM layer with 128 units
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display the model architecture and summary
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Part D: Model Training and Sentiment Prediction

### 1. Train the LSTM model

We will train the model using the prepared training dataset (`X_train`, `Y_train`) and validate it using the validation dataset (`X_val`, `Y_val`). We'll specify the number of epochs and the batch size.

In [ ]:
EPOCHS = 5 # You can increase this for better performance, but 5 is good for initial run
BATCH_SIZE = 64

history = model.fit(
    X_train,
    Y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, Y_val)
)

print("\nModel training complete.")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 205s 653ms/step - accuracy: 0.9413 - loss: 0.1589 - val_accuracy: 0.8692 - val_loss: 0.3376
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 198s 634ms/step - accuracy: 0.9632 - loss: 0.1072 - val_accuracy: 0.8698 - val_loss: 0.4115
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 197s 629ms/step - accuracy: 0.9724 - loss: 0.0810 - val_accuracy: 0.8528 - val_loss: 0.4579
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 199s 634ms/step - accuracy: 0.9641 - loss: 0.0967 - val_accuracy: 0.7922 - val_loss: 0.5744
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 197s 631ms/step - accuracy: 0.9617 - loss: 0.1039 - val_accuracy: 0.8460 - val_loss: 0.5951

Model training complete.


### 2. Record and visualize training history

We'll extract the training and validation accuracy and loss from the `history` object and then visualize them using Matplotlib to observe the model's learning progress.

In [ ]:
import matplotlib.pyplot as plt

# Get training and validation metrics
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

### 3. Evaluate the trained model using the test dataset

After training, it's crucial to evaluate the model's performance on unseen data (the test set) to ensure it generalizes well.

In [ ]:
loss, accuracy = model.evaluate(x_test_padded, y_test, verbose=1)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

### 4. Predict the sentiment of sample reviews and calculate evaluation metrics

Let's pick a few reviews from the test set, predict their sentiment, and compare with the actual labels. We'll also calculate more detailed metrics like precision, recall, F1-score, and display a confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test set
y_pred_probs = model.predict(x_test_padded)
y_pred = (y_pred_probs > 0.5).astype(int)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Display sample predictions
print("\nSample Predictions:")
for i in range(10):
    original_review = decode_review(x_test[i])
    predicted_sentiment = "Positive" if y_pred[i] == 1 else "Negative"
    actual_sentiment = "Positive" if y_test[i] == 1 else "Negative"
    print(f"--- Review {i+1} ---")
    print(f"Review: {original_review[:150]}...") # Truncate for brevity
    print(f"Predicted: {predicted_sentiment} (Probability: {y_pred_probs[i][0]:.2f})")
    print(f"Actual: {actual_sentiment}")
    print("\n")

782/782 ━━━━━━━━━━━━━━━━━━━━ 99s 126ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82     12500
           1       0.79      0.92      0.85     12500

    accuracy                           0.84     25000
   macro avg       0.85      0.84      0.84     25000
weighted avg       0.85      0.84      0.84     25000


Confusion Matrix:
[[ 9409  3091]
 [  991 11509]]

Sample Predictions:
--- Review 1 ---
Review: ? please give this one a miss br br ? ? and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how mich...
Predicted: Negative (Probability: 0.18)
Actual: Negative


--- Review 2 ---
Review: ? this film requires a lot of patience because it focuses on mood and character development the plot is very simple and many of the scenes take place ...
Predicted: Positive (Probability: 1.00)
Actual: Positive


--- Review 3 ---
Review: ? many animation buffs consider ?